# 📊 PTA Treasurer Report Generator — v4
### File-name driven · Auto-detects month · Updates correct column · GitHub integration
---
**Monthly workflow:**
1. Drop files in input folders (`quickbooks_february_2026.csv`, `givebacks_february.csv`, bank PDF)
2. Run Cells 0–10 to generate the Excel report
3. Run Cell 11 to push code changes to GitHub

**File naming:**
- QuickBooks → `quickbooks_<month>_<year>.csv`  e.g. `quickbooks_february_2026.csv`
- Givebacks  → `givebacks_<month>.csv`           e.g. `givebacks_february.csv`
- Bank PDF   → any name (month detected from content)


## Cell 0 — Install Dependencies
*Run once.*

In [1]:
import subprocess, sys, platform
print(f'Python: {sys.version}')
print(f'OS: {platform.system()}')

print('\nInstalling Python packages...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'openpyxl', 'pdfplumber', 'playwright', 'python-dotenv', '-q'])
print('  Done')

print('\nInstalling Playwright browser...')
try:
    r = subprocess.run(
        [sys.executable, '-m', 'playwright', 'install', 'chromium', '--with-deps'],
        capture_output=True, text=True)
    if r.returncode != 0:
        r2 = subprocess.run(
            [sys.executable, '-m', 'playwright', 'install', 'chromium'],
            capture_output=True, text=True)
        if r2.returncode != 0:
            print('  WARNING: Playwright install failed.')
            print('  Skip Cell 2 and upload your Givebacks CSV manually.')
        else:
            print('  Done')
    else:
        print('  Done')
except Exception as e:
    print(f'  WARNING: {e}')
    print('  Skip Cell 2 and upload your Givebacks CSV manually.')

print('\nAll dependencies ready.')


Python: 3.8.5 (default, Sep  4 2020, 02:22:02) 
[Clang 10.0.0 ]
OS: Darwin

Installing Python packages...
  Done

Installing Playwright browser...
  Done

All dependencies ready.


## Cell 1 — Configuration
*Set credentials. Month is auto-detected from filenames.*

In [59]:
from pathlib import Path
from datetime import datetime
import os, re, json, csv
from dotenv import load_dotenv

load_dotenv()

# Organisation name
ORG_NAME = os.getenv('ORG_NAME', 'Setauket School PTA')
INPUT_MONTH = "April"
# Input / output folders
GB_FOLDER   = Path('input/givebacks')/INPUT_MONTH
QB_FOLDER   = Path('input/quickbooks')
BANK_FOLDER = Path('input/bank')
HISTORY_DIR = Path('data/history')

# Givebacks credentials (set here or in .env)
GIVEBACKS_EMAIL    = os.getenv('GIVEBACKS_EMAIL',    '')
GIVEBACKS_PASSWORD = os.getenv('GIVEBACKS_PASSWORD', '')
#GIVEBACKS_URL      = os.getenv('GIVEBACKS_URL',      'https://app.mygivebacks.com')
GIVEBACKS_ORG_URL = os.getenv('GIVEBACKS_ORG_URL', '')

# Prompt once if not set
if not GIVEBACKS_ORG_URL:
    GIVEBACKS_ORG_URL = input(
        'Enter your Givebacks organisation URL\n'
        '(e.g. https://yourschool.givebacks.com): '
    ).strip().rstrip('/')
    print(f'Using: {GIVEBACKS_ORG_URL}')
    print('Tip: add GIVEBACKS_ORG_URL to your .env file to skip this prompt next time.')

# GitHub settings (set here or in .env)
GITHUB_REMOTE = os.getenv('GITHUB_REMOTE', 'origin')   # remote name
GITHUB_BRANCH = os.getenv('GITHUB_BRANCH', 'main')     # branch to push to

# Fiscal year: July = index 0, June = index 11
FISCAL_MONTHS      = ['JULY','AUG','SEPT','OCT','NOV','DEC','JAN','FEB','MAR','APR','MAY','JUNE']
FISCAL_START_MONTH = 7   # July

MONTH_NAMES = {
    'january':0,'february':1,'march':2,'april':3,'may':4,'june':5,
    'july':6,'august':7,'september':8,'october':9,'november':10,'december':11,
    'jan':0,'feb':1,'mar':2,'apr':3,'jun':5,
    'jul':6,'aug':7,'sep':8,'oct':9,'nov':10,'dec':11
}

def calendar_to_fiscal(cal_month_num):
    return (cal_month_num - FISCAL_START_MONTH) % 12

def month_name_to_fiscal_index(month_name):
    cal_idx = MONTH_NAMES.get(month_name.lower())
    if cal_idx is None: return None
    return calendar_to_fiscal(cal_idx + 1)

for folder in [GB_FOLDER, QB_FOLDER, BANK_FOLDER, Path('output'), HISTORY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Organisation : {ORG_NAME}')
print(f'Fiscal year  : July (index 0) -> June (index 11)')
print(f'Givebacks    : {"credentials set" if GIVEBACKS_EMAIL else "no credentials - manual upload"}')
print(f'GitHub       : remote={GITHUB_REMOTE}  branch={GITHUB_BRANCH}')


Organisation : Setauket School PTA
Fiscal year  : July (index 0) -> June (index 11)
Givebacks    : credentials set
GitHub       : remote=origin  branch=main


In [60]:
#from pathlib import Path
#session_file = Path('data/browser_session/givebacks_session.json')
#if session_file.exists():
#    session_file.unlink()
#    print('Session cleared')
#else:
#    print('No session file found')

In [61]:
#from pathlib import Path
#session_dir = Path('data/browser_session')
#print('Folder exists:', session_dir.exists())
#print('Contents:', list(session_dir.glob('*')) if session_dir.exists() else 'N/A')

## Cell 2 — Auto-Download Givebacks *(optional)*
Skip if uploading manually. Requires credentials in Cell 1 or `.env`.

In [62]:
import asyncio, calendar as cal_lib

async def download_givebacks(month_label, email, password, org_url, dest_folder):
    from playwright.async_api import async_playwright, TimeoutError as PWTimeout
    import csv as csv_module
    from datetime import datetime

    month_short  = month_label.split()[0].lower()   # e.g. 'april'
    year         = month_label.split()[1]            # e.g. '2026'
    target_month = datetime.strptime(month_label, '%B %Y').month
    target_year  = int(year)

    # Check if already downloaded
    existing = list(dest_folder.glob(f'givebacks_{month_short}*.csv'))
    if existing:
        print(f'Already downloaded: {len(existing)} file(s) for {month_label}')
        return existing

    print(f'Opening browser for {month_label}...')

    # Load saved session if available (avoids OTP after first run)
    session_dir  = Path('data/browser_session')
    session_file = session_dir / 'givebacks_session.json'
    session_dir.mkdir(parents=True, exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage'])

        # Load existing session if saved
        ctx = await browser.new_context(
            storage_state=str(session_file) if session_file.exists() else None
        )
        page = await ctx.new_page()

        try:
            # ── Check if already logged in via saved session ───────────────
            print('  -> Checking session...')
            await page.goto(f'{org_url}/payouts', timeout=60000)
            await page.wait_for_load_state('domcontentloaded')
            await page.wait_for_timeout(3000)  # ← give page more time to settle
            
            # Dismiss survey popup if present
            try:
                await page.locator('[aria-label="Close"], button:has-text("×")').first.click(timeout=2000)
                print('  Dismissed popup')
            except:
                await page.keyboard.press('Escape')
            await page.wait_for_timeout(500)
            
            # If redirected to login, need to authenticate
            current_url = page.url
            print(f'  Landing URL: {current_url}')
            login_form  = await page.locator('input[type="password"]').count()
            needs_login = ('login' in current_url.lower() or
                          'sign-in' in current_url.lower() or    # ← already there
                          'one-time-passcode' in current_url or
                          login_form > 0)

            if needs_login:
                print('  -> Logging in...')
                # Site redirected to sign-in — stay on that page instead of navigating to /login
                await page.goto(f'{org_url}/sign-in', timeout=30000)
                await page.wait_for_load_state('domcontentloaded')
                #await page.screenshot(path='debug_login.png')

                # Fill email
                await page.wait_for_selector(
                    'input[type="email"], input[placeholder*="email" i]',
                    timeout=10000)
                await page.fill(
                    'input[type="email"], input[placeholder*="email" i]', email)

                # Fill password
                await page.fill(
                    'input[type="password"], input[name="password"]', password)

                # Click Sign In — exact match to avoid Apple/Google buttons
                buttons = await page.locator('button').all()
                for btn in buttons:
                    txt = (await btn.inner_text()).strip()
                    if txt == 'Sign In':
                        await btn.click()
                        break
                await page.wait_for_load_state('domcontentloaded')
                await page.wait_for_timeout(2000)

                # ── Handle OTP if shown ────────────────────────────────────
                if 'one-time-passcode' in page.url:
                    print('  OTP required - check your email')
                    code_val = input('  Enter the 6-digit code: ').strip()
                    
                    # Type each digit into its own box
                    otp_boxes = await page.locator('input').all()
                    # Filter to only visible input boxes
                    visible_boxes = []
                    for box in otp_boxes:
                        if await box.is_visible():
                            visible_boxes.append(box)
                    
                    print(f'  Found {len(visible_boxes)} input boxes')
                    for i, digit in enumerate(code_val[:6]):
                        if i < len(visible_boxes):
                            await visible_boxes[i].click()
                            await visible_boxes[i].type(digit)
                            await page.wait_for_timeout(200)
                    
                    await page.wait_for_timeout(1000)
                    
                    # Check "Trust this browser for 60 days"
                    trust = page.locator('input[type="checkbox"]')
                    if await trust.count() > 0:
                        await trust.click()
                        print('  Checked: Trust this browser for 60 days')
                    
                    # Wait for Submit to be enabled then click
                    await page.wait_for_selector(
                        'button:has-text("Submit"):not([disabled])',
                        timeout=15000)
                    await page.click('button:has-text("Submit")')
                    await page.wait_for_load_state('domcontentloaded')
                    await page.wait_for_timeout(2000)
                    print('  OTP verified')

                # Verify login succeeded
                print(f'  URL after login: {page.url}')
                await page.screenshot(path='debug_after_login.png')
                if 'one-time-passcode' in page.url or 'login' in page.url.lower():
                    raise Exception('Login failed - OTP may be incorrect or expired')
                print('  Logged in successfully')

                # Save session so future runs skip login + OTP
                await ctx.storage_state(path=str(session_file))
                print(f'  Session saved -> {session_file}')

                # Navigate to payouts now that we are logged in
                await page.goto(f'{org_url}/payouts', timeout=60000)
                await page.wait_for_load_state('domcontentloaded')
                await page.wait_for_timeout(2000)
            
                # Dismiss any popup/survey that may be blocking the page
                try:
                    close_btn = page.locator('button:has-text("×"), [aria-label="Close"], button:has-text("close")').first
                    if await close_btn.is_visible():
                        await close_btn.click()
                        print('  Dismissed popup')
                except:
                    pass
            # Also try pressing Escape
            await page.keyboard.press('Escape')
            await page.wait_for_timeout(500)
            
            await page.screenshot(path='debug_payouts.png')
            #else:
            #    print('  Session valid - skipping login')

            # ── Find all payouts for target month ──────────────────────────
            print(f'  -> Finding payouts for {month_label}...')

            rows = await page.locator('table tbody tr').all()
            if not rows:
                rows = await page.locator('tr').all()

            payout_urls = []
            for row in rows:
                row_text = await row.inner_text()
                if str(target_year) not in row_text:
                    continue
                # Match month number e.g. "4/" or "04/"
                month_found = any(
                    fmt in row_text
                    for fmt in [f'{target_month}/', f'0{target_month}/']
                )
                if not month_found:
                    continue
                # Try to get payout ID from href
                links = await row.locator('a').all()
                for link in links:
                    href = await link.get_attribute('href')
                    if href and 'payouts/po_' in href:
                        payout_id  = href.split('payouts/')[1].split('/')[0]
                        summary_url = f'{org_url}/payouts/{payout_id}/summary'
                        payout_urls.append((payout_id, summary_url))
                        break
                else:
                    # Fallback: extract payout ID from row text
                    pid = re.search(r'po_\w+', row_text)
                    if pid:
                        payout_id  = pid.group(0)
                        summary_url = f'{org_url}/payouts/{payout_id}/summary'
                        payout_urls.append((payout_id, summary_url))

            # Deduplicate
            seen = set()
            payout_urls = [
                x for x in payout_urls
                if x[0] not in seen and not seen.add(x[0])
            ]
            print(f'  Found {len(payout_urls)} payout(s) for {month_label}')

            if not payout_urls:
                print(f'  WARNING: No payouts found for {month_label}')
                print(f'  Check debug_payouts.png to see the page')
                return []

            # ── Scrape each payout summary page ───────────────────────────
            saved_files = []
            for i, (payout_id, summary_url) in enumerate(payout_urls, 1):
                print(f'  -> Scraping payout {i}/{len(payout_urls)}: {payout_id}')
                await page.goto(summary_url, timeout=60000)
                await page.wait_for_load_state('domcontentloaded')
                await page.wait_for_timeout(2000)  # ← give table time to render

                # DEBUG: screenshot first payout only
                #if i == 1:
                #    await page.screenshot(path='debug_summary.png')
                #   # Also print all table-like elements found
                #    tables = await page.locator('table').count()
                #    rows_found = await page.locator('tr').count()
                #    print(f'    Tables found: {tables}  Rows found: {rows_found}')
                #    # Print page text to see structure
                #    body = await page.locator('body').inner_text()
                #    print(f'    Page text preview: {body[:500]}')
                    
                rows_data = []
                table_rows = await page.locator('table tbody tr').all()
                for tr in table_rows:
                    cells = await tr.locator('td').all()
                    if len(cells) < 3:
                        continue
                    item = (await cells[0].inner_text()).strip()
                    if not item or item in ('Item', 'Total'):
                        continue
                    category = (await cells[1].inner_text()).strip()
                    try:    txns = int((await cells[2].inner_text()).strip())
                    except: txns = 0
                    try:
                        total = float(
                            (await cells[3].inner_text()).strip()
                            .replace('$','').replace(',',''))
                    except: total = 0.0
                    rows_data.append({
                        'Item':                item,
                        'Categories':          category,
                        'No. of Transactions': txns,
                        'Total':               f'${total:,.2f}',
                    })

                if not rows_data:
                    print(f'    WARNING: No data scraped from {summary_url}')
                    continue

                dest_folder.mkdir(parents=True, exist_ok=True)
                fname = dest_folder / f'givebacks_{month_short}_{payout_id}.csv'
                with open(fname, 'w', newline='') as f:
                    writer = csv_module.DictWriter(
                        f, fieldnames=[
                            'Item','Categories','No. of Transactions','Total'])
                    writer.writeheader()
                    writer.writerows(rows_data)
                print(f'    Saved: {fname.name}  ({len(rows_data)} items)')
                saved_files.append(fname)

            print(f'\n  Done: {len(saved_files)} CSV file(s) saved to {dest_folder}')
            return saved_files

        except Exception as e:
            await page.screenshot(path='debug_error.png')
            raise Exception(f'{e}\nCheck debug_error.png for page state')
        finally:
            await ctx.close()
            await browser.close()

# Use INPUT_MONTH directly as the source of truth
year_str = str(datetime.today().year)
label    = f'{INPUT_MONTH.capitalize()} {year_str}'
print(f'Downloading Givebacks for: {label}  (from INPUT_MONTH in Cell 1)')

            # ── Run ───────────────────────────────────────────────────────────────────────
if not GIVEBACKS_EMAIL or not GIVEBACKS_PASSWORD:
    print('No credentials found.')
    print('Set GIVEBACKS_EMAIL and GIVEBACKS_PASSWORD in your .env file')
    print('OR place CSV files manually in:', GB_FOLDER)
else:
    try:
        await download_givebacks(
            label, GIVEBACKS_EMAIL, GIVEBACKS_PASSWORD,
            GIVEBACKS_ORG_URL, GB_FOLDER)
    except Exception as e:
        print(f'Download failed: {e}')
        print('Upload Givebacks CSV manually to', GB_FOLDER)

Already downloaded: 4 file(s) for April 2026


## Cell 3 — Get Consistent Files across all three Platforms for a given month
Reads filenames and PDF content to determine which fiscal month to update.


In [63]:
GB_FOLDER

PosixPath('input/givebacks/April')

In [64]:
import pdfplumber
def detect_month_from_filename(filepath):
    name  = filepath.stem.lower()
    parts = re.split(r'[_\-\s]+', name)
    month_str = None; year_str = None; fiscal_idx = None
    for part in parts:
        if part in MONTH_NAMES and month_str is None:
            month_str  = part
            fiscal_idx = month_name_to_fiscal_index(part)
        if re.match(r'^20\d{2}$', part) and year_str is None:
            year_str = part
    if month_str is None: return None, None
    year_str    = year_str or str(datetime.today().year)
    month_label = f'{month_str.capitalize()} {year_str}'
    return month_label, fiscal_idx

def detect_month_from_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = pdf.pages[0].extract_text() or ''
        m = re.search(
            r'(January|February|March|April|May|June|July|August|'
            r'September|October|November|December)\s+(\d{1,2}),\s+(\d{4})'
            r'\s+through', text, re.IGNORECASE)
        if m:
            return f'{m.group(1)} {m.group(3)}', month_name_to_fiscal_index(m.group(1))
    except Exception as e:
        print(f'  Could not read PDF: {e}')
    return None, None

# QuickBooks
qb_files = sorted(f for f in QB_FOLDER.glob('*.csv') 
                  if INPUT_MONTH.lower() in f.name.lower())
if not qb_files:
    all_qb = sorted(QB_FOLDER.glob('*.csv'))
    raise FileNotFoundError(
        f'No QuickBooks file found for {INPUT_MONTH}\n'
        f'Files available: {[f.name for f in all_qb]}\n'
        f'Expected: quickbooks_{INPUT_MONTH.lower()}_2026.csv'
    )
QB_FILE = qb_files[-1]

QB_MONTH_LABEL, QB_FISCAL_IDX = detect_month_from_filename(QB_FILE)

print(f'QuickBooks : {QB_FILE.name}  ->  {QB_MONTH_LABEL}  (fiscal index {QB_FISCAL_IDX}: {FISCAL_MONTHS[QB_FISCAL_IDX]})')
if QB_MONTH_LABEL.split()[0].lower() != INPUT_MONTH.lower():
    raise ValueError(
        f'\nMONTH MISMATCH - QuickBooks file!\n'
        f'  INPUT_MONTH set to : {INPUT_MONTH}\n'
        f'  QB file detected   : {QB_MONTH_LABEL}\n'
        f'  Fix: rename file to quickbooks_{INPUT_MONTH.lower()}_2026.csv\n'
        f'       or update INPUT_MONTH in Cell 1'
    )
print(f'  Month matches INPUT_MONTH ✅')

# Givebacks
#gb_files = sorted(GB_FOLDER.glob('givebacks_*.csv'))
gb_files = sorted(f for f in GB_FOLDER.rglob('*.csv') if 'givebacks' in f.name.lower())
if not gb_files:
    raise FileNotFoundError(
        f'No Givebacks files in {GB_FOLDER}\nName them: givebacks_february.csv')
GB_FILE_INFO = []
print(f'\nGivebacks :')
for f in gb_files:
    lbl, idx = detect_month_from_filename(f)
    if lbl:
        print(f'  {f.name}  ->  {lbl}  (fiscal index {idx})')
        GB_FILE_INFO.append((f, lbl, idx))
    else:
        print(f'  {f.name}  ->  WARNING: could not detect month, skipping')

# Validate all Givebacks files match INPUT_MONTH
for f, lbl, idx in GB_FILE_INFO:
    if lbl.split()[0].lower() != INPUT_MONTH.lower():
        raise ValueError(
            f'\nMONTH MISMATCH - Givebacks file!\n'
            f'  INPUT_MONTH set to  : {INPUT_MONTH}\n'
            f'  File {f.name} detected as: {lbl}\n'
            f'  Fix: move file to correct month folder\n'
            f'       or update INPUT_MONTH in Cell 1'
        )
print(f'  All Givebacks files match INPUT_MONTH ✅')
        
# Bank PDF
pdf_files = sorted(BANK_FOLDER.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF in {BANK_FOLDER}')
BANK_FILE = pdf_files[-1]
BANK_MONTH_LABEL, BANK_FISCAL_IDX = detect_month_from_pdf(BANK_FILE)
print(f'\nBank PDF  : {BANK_FILE.name}  ->  ', end='')
if BANK_MONTH_LABEL:
    print(f'{BANK_MONTH_LABEL}  (fiscal index {BANK_FISCAL_IDX})')
else:
    BANK_MONTH_LABEL, BANK_FISCAL_IDX = QB_MONTH_LABEL, QB_FISCAL_IDX
    print(f'month not detected - using QB month: {QB_MONTH_LABEL}')

if BANK_MONTH_LABEL and BANK_MONTH_LABEL.split()[0].lower() != INPUT_MONTH.lower():
    raise ValueError(
        f'\nMONTH MISMATCH - Bank PDF!\n'
        f'  INPUT_MONTH set to : {INPUT_MONTH}\n'
        f'  Bank PDF detected  : {BANK_MONTH_LABEL}\n'
        f'  Fix: check you uploaded the correct month statement'
    )
if BANK_MONTH_LABEL:
    print(f'  Bank month matches INPUT_MONTH ✅')
    
MONTH_LABEL = QB_MONTH_LABEL
FISCAL_IDX  = QB_FISCAL_IDX
safe_month  = MONTH_LABEL.replace(' ', '_')
OUTPUT_FILE = Path(f'output/Treasurer_Report_{safe_month}.xlsx')

print(f" {'='*55}")
print(f'  Report month : {MONTH_LABEL}')
print(f'  Fiscal index : {FISCAL_IDX}  -> column {FISCAL_MONTHS[FISCAL_IDX]} in budget sheets')
print(f'  Output       : {OUTPUT_FILE}')
print(f"{'='*55}")


QuickBooks : quickbooks_april_2026.csv  ->  April 2026  (fiscal index 9: APR)
  Month matches INPUT_MONTH ✅

Givebacks :
  givebacks_april_1.csv  ->  April 2026  (fiscal index 9)
  givebacks_april_2.csv  ->  April 2026  (fiscal index 9)
  givebacks_april_3.csv  ->  April 2026  (fiscal index 9)
  givebacks_april_4.csv  ->  April 2026  (fiscal index 9)
  All Givebacks files match INPUT_MONTH ✅

Bank PDF  : Chase_march_statement.pdf  ->  month not detected - using QB month: April 2026
  Bank month matches INPUT_MONTH ✅
  Report month : April 2026
  Fiscal index : 9  -> column APR in budget sheets
  Output       : output/Treasurer_Report_April_2026.xlsx


In [65]:
print('QB_FILE exists:  ', QB_FILE.exists())
print('GB files found:  ', list(GB_FOLDER.glob('*.csv')))
print('Bank files found:', list(BANK_FOLDER.glob('*.pdf')))

QB_FILE exists:   True
GB files found:   [PosixPath('input/givebacks/April/givebacks_april_1.csv'), PosixPath('input/givebacks/April/givebacks_april_2.csv'), PosixPath('input/givebacks/April/givebacks_april_3.csv'), PosixPath('input/givebacks/April/givebacks_april_4.csv')]
Bank files found: [PosixPath('input/bank/Chase_april_2026.pdf'), PosixPath('input/bank/Chase_march_statement.pdf')]


## Cell 4 — Parse Files & Save to History

In [111]:
from collections import defaultdict

try:
    qb = parse_quickbooks_detail(QB_FOLDER)
    print('QB OK:', qb['income_total'])
except Exception as e:
    print('QB FAILED:', e)

# Step 2 - test Givebacks alone
try:
    givebacks = parse_givebacks_files(GB_FILE_INFO)
    print('Givebacks OK:', len(givebacks))
except Exception as e:
    print('Givebacks FAILED:', e)

# Step 3 - test Bank alone
try:
    bank = parse_chase_pdf(BANK_FILE)
    print('Bank OK:', bank['beginning_balance'])
except Exception as e:
    print('Bank FAILED:', e)

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
safe_month = MONTH_LABEL.replace(' ', '_')

def parse_quickbooks_detail(folder):
    """
    Parses QuickBooks Transaction Detail by Account export.
    Returns same structure as parse_quickbooks() for compatibility,
    plus detailed transactions list for Debits/Credits sheet.
    """
    files = sorted(f for f in folder.glob('*.csv')
                   if INPUT_MONTH.lower() in f.name.lower())
    if not files:
        all_files = sorted(folder.glob('*.csv'))
        raise FileNotFoundError(
            f'No QuickBooks detail file found for {INPUT_MONTH}\n'
            f'Files available: {[f.name for f in all_files]}\n'
            f'Expected name containing: {INPUT_MONTH.lower()}'
        )
    path = files[-1]
    print(f'  QuickBooks detail: {path.name}')

    # Read with encoding fallback
    lines = []
    for encoding in ['utf-8-sig', 'utf-16', 'utf-8', 'latin-1', 'windows-1252']:
        try:
            with open(path, 'rb') as f:
                raw = f.read()
            content = raw.replace(b'\x00', b'').decode(encoding)
            content = content.replace('\r\n', '\n').replace('\r', '\n')
            reader  = csv.reader(content.splitlines())
            lines   = [row for row in reader if any(c.strip() for c in row)]
            if lines:
                print(f'  Encoding: {encoding}')
                break
        except (UnicodeDecodeError, UnicodeError):
            continue
    if not lines:
        raise ValueError(f'Could not read {path.name}')

    data = {
        'period':        '',
        'income':        {},   # category -> total amount
        'income_total':  0.0,
        'expenses':      {},   # category -> total amount
        'expense_total': 0.0,
        'net_income':    0.0,
        'transactions':  [],   # detailed list for Debits/Credits sheet
    }

    # Extract period from line 3
    if len(lines) > 2:
        data['period'] = lines[2][0].strip().strip('"')

    # Skip header rows — find the column header row
    header_row_idx = None
    for i, row in enumerate(lines):
        if 'Transaction date' in row or 'Transaction type' in row:
            header_row_idx = i
            break

    if header_row_idx is None:
        raise ValueError('Could not find header row in QuickBooks detail file')

    # Column indices
    headers = lines[header_row_idx]
    def col(name):
        for i, h in enumerate(headers):
            if name.lower() in h.lower():
                return i
        return None

    idx_date   = col('Transaction date') or 1
    idx_type   = col('Transaction type') or 2
    idx_num    = col('Num')              or 3
    idx_name   = col('Name')             or 4
    idx_desc   = col('Description')      or 5
    idx_split  = col('Split')            or 6
    idx_amount = col('Amount')           or 7

    def parse_amount(val):
        try:
            return float(str(val).replace('$','').replace(',','').replace('"','').strip())
        except:
            return 0.0

    # Parse category sections
    # Skip the "Checking (4346)" section — use category sections only
    current_category = None
    in_checking      = False
    in_total_section = False

    SKIP_SECTIONS = {'checking', 'total', 'accrual'}
    PARENT_SECTIONS = {
        'fundraising-net', 'program-income', 'program expense',
        'graduating class dues', 'admin/general'
    }

    for row in lines[header_row_idx + 1:]:
        if len(row) < 2:
            continue

        first = row[0].strip().strip('"')
        second = row[1].strip().strip('"') if len(row) > 1 else ''

        # Detect section headers — non-empty first col, empty second col
        if first and not second:
            first_lower = first.lower()
            # Skip checking account section and total lines
            if 'checking' in first_lower and '4346' in first_lower:
                in_checking = True
                current_category = None
                continue
            if first_lower.startswith('total for checking'):
                in_checking = False
                continue
            if first_lower.startswith('total for') or first_lower == 'total':
                continue
            if any(s in first_lower for s in SKIP_SECTIONS):
                continue
            if any(s in first_lower for s in PARENT_SECTIONS):
                # Parent section header — don't set as category
                continue
            if 'accrual basis' in first_lower:
                continue
            # This is a real category section header
            in_checking = False
            current_category = first
            continue

        # Skip rows while in Checking section
        if in_checking:
            continue

        # Skip if no category set
        if not current_category:
            continue

        # Skip total rows
        if first.lower().startswith('total'):
            continue

        # Parse transaction row — date is in second column
        date_val   = row[idx_date].strip()   if len(row) > idx_date   else ''
        type_val   = row[idx_type].strip()   if len(row) > idx_type   else ''
        num_val    = row[idx_num].strip()    if len(row) > idx_num    else ''
        name_val   = row[idx_name].strip()   if len(row) > idx_name   else ''
        desc_val   = row[idx_desc].strip()   if len(row) > idx_desc   else ''
        amt_val    = row[idx_amount].strip() if len(row) > idx_amount  else ''

        if not date_val or not amt_val:
            continue

        amount = parse_amount(amt_val)
        if amount == 0.0:
            continue

        # Clean up description — remove long bank trace numbers
        clean_desc = desc_val
        if 'ORIG CO NAME' in desc_val:
            clean_desc = 'MemberHub/Givebacks Deposit'
        elif desc_val.startswith('CHECK #'):
            clean_desc = desc_val

        # In category sections: Deposit = income, Check/Expense = expense
        is_deposit = type_val.lower() == 'deposit'
        is_check   = type_val.lower() in ('check', 'expense', 'bill payment', 'bill')
        
        transaction = {
            'date':        date_val,
            'type':        type_val,
            'check_no':    num_val,
            'payee':       name_val,
            'description': clean_desc,
            'category':    current_category,
            'amount':      abs(amount),
            'is_income':   is_deposit,   # ← now correctly based on type only
        }
        data['transactions'].append(transaction)

        # Accumulate into income or expense totals
        if is_deposit:
            data['income'][current_category] = (
                data['income'].get(current_category, 0.0) + abs(amount))
        else:
            data['expenses'][current_category] = (
                data['expenses'].get(current_category, 0.0) + abs(amount))

    # Calculate totals
    data['income_total']  = sum(data['income'].values())
    data['expense_total'] = sum(data['expenses'].values())
    data['net_income']    = data['income_total'] - data['expense_total']

    print(f'  Period        : {data["period"]}')
    print(f'  Income items  : {len(data["income"])}  total=${data["income_total"]:,.2f}')
    print(f'  Expense items : {len(data["expenses"])}  total=${data["expense_total"]:,.2f}')
    print(f'  Transactions  : {len(data["transactions"])}')
    return data

def parse_givebacks_files(file_info_list):
    merged = {}
    for fpath, lbl, idx in file_info_list:
        with open(fpath, newline='', encoding='utf-8-sig') as f:
            for r in csv.DictReader(f):
                item = r.get('Item','').strip()
                if not item: continue
                try:    amt = float(str(r.get('Total','0')).replace('$','').replace(',',''))
                except: amt = 0.0
                try:    cnt = int(r.get('No. of Transactions','0').strip() or 0)
                except: cnt = 0
                if item in merged:
                    merged[item]['count'] += cnt
                    merged[item]['total'] += amt
                    merged[item]['sources'].add(fpath.name)
                else:
                    merged[item] = {'item':item,'category':r.get('Categories','').strip(),
                                    'count':cnt,'total':amt,'sources':{fpath.name}}
    rows = list(merged.values())
    for r in rows: r['source_file'] = ', '.join(sorted(r['sources'])); del r['sources']
    return rows

def parse_chase_pdf(path):
    bank = {'period':'','account':'','beginning_balance':0.0,'ending_balance':0.0,
            'total_deposits':0.0,'total_checks':0.0,'total_fees':0.0,
            'total_withdrawals':0.0,                                          # ← new
            'deposits':[],'checks':[],'fees':[],'withdrawals':[],             # ← new
            'daily_balances':{},'source_file':path.name}
    with pdfplumber.open(path) as pdf:
        text = chr(10).join(p.extract_text() or '' for p in pdf.pages)
    def grab(pat):
        m = re.search(pat, text)
        try: return float(m.group(1).replace(',','')) if m else 0.0
        except: return 0.0
    m = re.search(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d+,\s+\d{4}\s+through\s+\S+\s+\d+,\s+\d{4}', text)
    if m: bank['period'] = m.group(0)
    m = re.search(r'Account Number:\s+([\d]+)', text)
    if m: bank['account'] = m.group(1)
    bank['beginning_balance'] = grab(r'Beginning Balance\s+\$?([\d,]+\.\d{2})')
    bank['ending_balance']    = grab(r'Ending Balance\s+\d+\s+\$?([\d,]+\.\d{2})')
    bank['total_deposits']    = grab(r'Total Deposits and Additions\s+\$?([\d,]+\.\d{2})')
    bank['total_checks']      = grab(r'Total Checks Paid\s+\$?([\d,]+\.\d{2})')
    bank['total_fees']        = grab(r'Total Fees\s+\$?([\d,]+\.\d{2})')
    bank['total_withdrawals'] = grab(r'Total Other Withdrawals\s+\$?([\d,]+\.\d{2})')  # ← new

    dep = re.search(r'DEPOSITS AND ADDITIONS(.*?)CHECKS PAID', text, re.DOTALL)
    if dep:
        for m in re.finditer(r'(\d{2}/\d{2})\s+(.+?)\s+\$?([\d,]+\.\d{2})', dep.group(1)):
            bank['deposits'].append({'date':m.group(1),'description':m.group(2).strip(),'amount':float(m.group(3).replace(',',''))})

    for m in re.finditer(r'(\d{4})\s+\^?\s+(\d{2}/\d{2})\s+\$?([\d,]+\.\d{2})', text):
        bank['checks'].append({'check_no':m.group(1),'date':m.group(2),'amount':float(m.group(3).replace(',',''))})

    fee = re.search(r'FEES(.*?)DAILY ENDING BALANCE', text, re.DOTALL)
    if fee:
        for m in re.finditer(r'(\d{2}/\d{2})\s+(.+?)\s+\$?([\d,]+\.\d{2})', fee.group(1)):
            bank['fees'].append({'date':m.group(1),'description':m.group(2).strip(),'amount':float(m.group(3).replace(',',''))})

    # ── Withdrawals section ────────────────────────────────────────────────── new
    wd = re.search(r'OTHER WITHDRAWALS(.*?)(?:FEES|DAILY ENDING BALANCE)', text, re.DOTALL | re.IGNORECASE)
    if wd:
        for m in re.finditer(r'(\d{2}/\d{2})\s+(.+?)\s+([\d,]+\.\d{2})', wd.group(1)):
            bank['withdrawals'].append({
                'date':        m.group(1),
                'description': m.group(2).strip(),
                'amount':      float(m.group(3).replace(',',''))
            })
    if bank['total_withdrawals'] == 0.0 and bank['withdrawals']:
        bank['total_withdrawals'] = sum(w['amount'] for w in bank['withdrawals'])
    # ──────────────────────────────────────────────────────────────────────────

    bal = re.search(r'DAILY ENDING BALANCE(.*)$', text, re.DOTALL)
    if bal:
        for m in re.finditer(r'(\d{2}/\d{2})\s+([\d,]+\.\d{2})', bal.group(1)):
            bank['daily_balances'][m.group(1)] = float(m.group(2).replace(',',''))
    return bank

print(f'QuickBooks : income=${qb["income_total"]:,.2f}  expenses=${qb["expense_total"]:,.2f}  net=${qb["net_income"]:,.2f}')
print(f'Givebacks  : {len(givebacks)} items  total=${sum(g["total"] for g in givebacks):,.2f}')
print(f'Bank       : beginning=${bank["beginning_balance"]:,.2f}  ending=${bank["ending_balance"]:,.2f}')

# Save to history

try:
    safe_month = MONTH_LABEL.replace(' ', '_')
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)
    hist_entry = {
        'month_label':     MONTH_LABEL,
        'fiscal_index':    FISCAL_IDX,
        'income':          qb['income'],
        'expenses':        qb['expenses'],
        'income_total':    qb['income_total'],
        'expense_total':   qb['expense_total'],
        'net_income':      qb['net_income'],
        'givebacks_total': sum(g['total'] for g in givebacks),
        'generated_at':    datetime.today().isoformat(),
    }
    # Always overwrite existing history for current month
    hist_path = HISTORY_DIR / f'{safe_month}.json'
    if hist_path.exists():
        print(f'  Overwriting existing history for {MONTH_LABEL}')
    hist_path.write_text(json.dumps(hist_entry, indent=2))
    print(f'History saved -> {hist_path}')
except Exception as e:
    print(f'\nWARNING: Could not save history: {e}')
    print('Report generation will continue.')


  QuickBooks detail: quickbooks_april_2026.csv
  Encoding: utf-8-sig
  Period        : April 1-30, 2026
  Income items  : 10  total=$31,732.00
  Expense items : 12  total=$21,534.79
  Transactions  : 48
QB OK: 31732.0
Givebacks OK: 20
Bank OK: 32203.66
QuickBooks : income=$31,732.00  expenses=$21,534.79  net=$10,197.21
Givebacks  : 20 items  total=$7,761.00
Bank       : beginning=$32,203.66  ending=$38,056.59
  Overwriting existing history for April 2026
History saved -> data/history/April_2026.json


In [98]:
import json
from pathlib import Path

hist_files = sorted(Path('data/history').glob('*.json'))
print(f'History files found: {len(hist_files)}')
for hf in hist_files:
    e = json.loads(hf.read_text())
    print(f'  {hf.name}')
    print(f'    month_label  : {e["month_label"]}')
    print(f'    fiscal_index : {e["fiscal_index"]}')
    print(f'    income_total : ${e["income_total"]:,.2f}')
    print(f'    expense_total: ${e["expense_total"]:,.2f}')

History files found: 2
  April_2026.json
    month_label  : April 2026
    fiscal_index : 9
    income_total : $31,732.00
    expense_total: $21,534.79
  March_2026.json
    month_label  : March 2026
    fiscal_index : 8
    income_total : $8,405.21
    expense_total: $2,552.28


In [99]:
print('Income actuals loaded:')
for item, vals in INCOME_ACTUALS_LIVE.items():
    non_zero = [(FISCAL_MONTHS[i], v) for i,v in enumerate(vals) if v != 0]
    if non_zero:
        print(f'  {item:<35} {non_zero}')

print('\nExpense actuals loaded:')
for item, vals in EXPENSE_ACTUALS_LIVE.items():
    non_zero = [(FISCAL_MONTHS[i], v) for i,v in enumerate(vals) if v != 0]
    if non_zero:
        print(f'  {item:<35} {non_zero}')

Income actuals loaded:
  Membership - Teachers               [('APR', 30.0)]
  Talent Show Income                  [('MAR', 660.0), ('APR', 200.0)]
  Bank Charges & Fees                 [('APR', 30.5)]
  Movie Night                         [('APR', 668.94)]
  Scholastic Book Fairs               [('APR', 5841.48)]
  Spring Dance K-5                    [('APR', 500.0)]
  Talent Show                         [('APR', 273.92)]
  Basket Dinner                       [('APR', 10500.0)]
  Contribution                        [('APR', 14390.0)]
  Plant Sale                          [('MAR', 130.0), ('APR', 1182.0)]
  Staff Appreciation                  [('APR', 1269.24)]
  Lawn Signs                          [('APR', 855.0)]
  Skate Night                         [('MAR', 40.0), ('APR', 20.0)]
  Birthday Books-Income               [('APR', 60.0)]
  Fast Athletics                      [('MAR', 4340.0), ('APR', 3360.0)]
  Arts In Education                   [('APR', 4045.0)]
  Basket dinner         

In [100]:
#try:
#    qb = parse_quickbooks(QB_FILE)
#    print('QB OK:', qb['income_total'])
#except Exception as e:
#    print('QB FAILED:', e)

## Step 2 - test Givebacks alone
#try:
#    givebacks = parse_givebacks_files(GB_FILE_INFO)
#    print('Givebacks OK:', len(givebacks))
#except Exception as e:
#    print('Givebacks FAILED:', e)

## Step 3 - test Bank alone
#try:
#    bank = parse_chase_pdf(BANK_FILE)
#    print('Bank OK:', bank['beginning_balance'])
#except Exception as e:
#    print('Bank FAILED:', e)

In [101]:
with open(QB_FILE, 'rb') as f:
    raw = f.read()

print(f'File size: {len(raw)} bytes')
print(f'First 200 bytes raw: {raw[:200]}')

cleaned = raw.replace(b'\x00', b'').decode('latin-1')
print(f'First 200 chars decoded: {cleaned[:200]}')

File size: 12059 bytes
First 200 bytes raw: b'SETAUKET SCHOOL PTA,,,,,,,,\r\nTransaction Detail by Account,,,,,,,,\r\n"April 1-30, 2026",,,,,,,,\r\n\r\n,Transaction date,Transaction type,Num,Name,Description,Split,Amount,Balance\r\nChecking (4346),,,,,,,,\r'
First 200 chars decoded: SETAUKET SCHOOL PTA,,,,,,,,
Transaction Detail by Account,,,,,,,,
"April 1-30, 2026",,,,,,,,

,Transaction date,Transaction type,Num,Name,Description,Split,Amount,Balance
Checking (4346),,,,,,,,


In [102]:
print(f'total_deposits    : ${bank["total_deposits"]:,.2f}')
print(f'total_checks      : ${bank["total_checks"]:,.2f}')
print(f'total_withdrawals : ${bank["total_withdrawals"]:,.2f}')
print(f'total_fees        : ${bank["total_fees"]:,.2f}')
print(f'beginning_balance : ${bank["beginning_balance"]:,.2f}')
print(f'ending_balance    : ${bank["ending_balance"]:,.2f}')
print(f'\nWithdrawal items found: {len(bank["withdrawals"])}')
for w in bank["withdrawals"]:
    print(f'  {w["date"]}  {w["description"][:45]:<46} ${w["amount"]:>10,.2f}')

calc = (bank['beginning_balance']
        + bank['total_deposits']
        - bank['total_checks']
        - bank.get('total_withdrawals', 0.0)
        - bank['total_fees'])
print(f'\nCalculated ending : ${calc:,.2f}')
print(f'Actual ending     : ${bank["ending_balance"]:,.2f}')
print(f'Difference        : ${calc - bank["ending_balance"]:,.2f}')

total_deposits    : $8,405.21
total_checks      : $2,552.28
total_withdrawals : $0.00
total_fees        : $0.00
beginning_balance : $32,203.66
ending_balance    : $38,056.59

Withdrawal items found: 0

Calculated ending : $38,056.59
Actual ending     : $38,056.59
Difference        : $0.00


## Cell 5 — Build Actuals from History
Assembles the 12-column actuals arrays from all stored monthly JSON files.

In [103]:
def load_all_actuals():
    hist_files = sorted(HISTORY_DIR.glob('*.json'))
    if not hist_files:
        print('No history yet - actuals will be zeros until months are processed.')
        return {}, {}
    income_actuals  = defaultdict(lambda: [0.0]*12)
    expense_actuals = defaultdict(lambda: [0.0]*12)
    for hf in hist_files:
        try:
            entry = json.loads(hf.read_text())
            idx   = entry.get('fiscal_index')
            if idx is None: continue
            for item, val in entry.get('income', {}).items():
                income_actuals[item][idx] = val
            for item, val in entry.get('expenses', {}).items():
                expense_actuals[item][idx] = val
            print(f'  Loaded: {hf.stem:<25} -> {entry["month_label"]:<18} (index {idx}: {FISCAL_MONTHS[idx]})')
        except Exception as e:
            print(f'  Warning: could not load {hf.name}: {e}')
    return dict(income_actuals), dict(expense_actuals)

print('Loading actuals from history...')
INCOME_ACTUALS_LIVE, EXPENSE_ACTUALS_LIVE = load_all_actuals()
print(f'\nIncome items tracked : {len(INCOME_ACTUALS_LIVE)}')
print(f'Expense items tracked: {len(EXPENSE_ACTUALS_LIVE)}')
print(f'\nValues for {MONTH_LABEL} (column {FISCAL_MONTHS[FISCAL_IDX]}):')
income_this_month = {k:v[FISCAL_IDX] for k,v in INCOME_ACTUALS_LIVE.items() if v[FISCAL_IDX]}
expense_this_month = {k:v[FISCAL_IDX] for k,v in EXPENSE_ACTUALS_LIVE.items() if v[FISCAL_IDX]}
print('  Income:')
for k,v in income_this_month.items():  print(f'    {k:<40} ${v:>10,.2f}')
print('  Expenses:')
for k,v in expense_this_month.items(): print(f'    {k:<40} ${v:>10,.2f}')


Loading actuals from history...
  Loaded: April_2026                -> April 2026         (index 9: APR)
  Loaded: March_2026                -> March 2026         (index 8: MAR)

Income items tracked : 11
Expense items tracked: 15

Values for April 2026 (column APR):
  Income:
    Membership - Teachers                    $     30.00
    Talent Show Income                       $    200.00
    Basket Dinner                            $ 10,500.00
    Contribution                             $ 14,390.00
    Plant Sale                               $  1,182.00
    Staff Appreciation                       $  1,135.00
    Lawn Signs                               $    855.00
    Skate Night                              $     20.00
    Birthday Books-Income                    $     60.00
    Fast Athletics                           $  3,360.00
  Expenses:
    Bank Charges & Fees                      $     30.50
    Movie Night                              $    668.94
    Scholastic Book Fairs 

## Cell 6 — Annual Budget Data
*Edit once per fiscal year (July). Last Year = prior year totals.*

In [104]:
# Maps QuickBooks category names -> Budget category names
QB_TO_BUDGET_MAP = {
    # Income
    'Fast Athletics':          'FAST',
    'Basket Dinner':           'Ticket & Raffle Sales',
    'Contribution':            'Ticket & Raffle Sales',
    'Skate Night':             'The Night at Rinx',
    'Birthday Books-Income':   'Birthday Books',
    'Membership - Teachers':   'Teachers',
    'Lawn Signs':              'Lawn Signs',
    'Plant Sale':              'Plant Sale',
    'Staff Appreciation':      'Staff Appreciation',
    'Talent Show Income':      'Talent Show',
    'Fall Pictures':           'Fall Pictures',
    # Expenses
    'Bank Charges & Fees':     'Bank Services',
    'Arts In Education':       'Cultural Arts',
    'Scholastic Book Fairs':   'Book Fair',
    'Movie Night':             'Outdoor Movie',
    'Basket dinner':           'Entertainment',
    'Basket Dinner - Venue':   'Venue',
    'Milk & Cookies':          'Milk & Cookies',
    'Spring Fling':            'Spring Fling',
    'Welcome Back Staff':      'Welcome Back Breakfast',
    'Gingerbread U':           'Gingerbread U',
    'Multicultural Night':     'Multicultural Night',
    'Science Fair':            'Science Fair',
    'Talent Show':             'Talent Show',
}

# Format: 'Item': (last_year_actual, annual_budget)
INCOME_BUDGET = {
    'Fundraising': {
        'Birthday Books':  (2310.00, 2000.00), 'Book Fair':       (9118.36,  500.00),
        'Croc Charms':     (405.00,   100.00), 'Fall Pictures':   (3350.25, 3000.00),
        'FAST':            (26370.00,1000.00), 'Holiday Boutique':(13417.00,7500.00),
        'Plant Sale':      (8202.68, 8000.00), 'Spiritwear':      (1253.96, 1500.00),
        'Spring Pictures': (0.00,       0.00),
    },
    'Basket Dinner': {
        'Ticket & Raffle Sales': (22165.00, 15000.00),
        'Sponsors':              (7450.00,   4000.00),
    },
    'Membership': {
        'Single':(2580.00,2000.00), 'Family':(1675.00,1000.00),
        'Donations':(290.63,100.00), 'Teachers':(0.00,165.00), 'Student':(0.00,5.00),
    },
    'Program': {
        'Gingerbread U':(3515.00,0.00), 'Staff Appreciation':(1625.00,1000.00),
        'Talent Show':(1070.00,750.00),
    },
    'Grad Class Activities': {
        'Family Contributions':(6480.00,3900.00), 'Lawn Signs':(2560.00,750.00),
        'Treat or Trunk':(1250.00,975.00), 'The Night at Rinx':(0.00,0.00),
    },
}

EXPENSE_BUDGET = {
    'Admin/General': {
        'Accountant':(650.00,650.00), 'Bank Services':(234.94,200.00),
        'Insurance':(350.00,350.00), 'Supplies':(450.22,500.00),
        'Accounting Quickbooks':(410.66,1300.00), 'Training':(68.90,100.00),
        'Website & Remind App':(344.01,1000.00), 'Event Equipment':(563.32,1000.00),
    },
    'Fundraising': {
        'Birthday Books':(503.39,500.00), 'Book Fair':(9523.91,10000.00),
        'Croc Charms':(205.00,0.00), 'Fall Pictures':(0.00,0.00),
        'FAST':(24000.00,17000.00), 'Holiday Boutique':(11714.86,10000.00),
        'Plant Sale':(5615.95,6000.00), 'Spiritwear':(0.00,1000.00),
        'Spring Pictures':(0.00,0.00),
    },
    'Membership': {
        'Council Dues':(125.00,150.00), 'Membership Expenses':(1034.00,1500.00),
    },
    'Basket Dinner': {
        'Entertainment':(1405.42,1000.00), 'Raffles':(1831.70,2000.00),
        'Venue':(8265.20,10000.00),
    },
    'Programs': {
        'Bus Driver Appreciation':(240.00,300.00), 'Electric Parade':(371.89,200.00),
        'Family Connect Nights':(771.00,1500.00), 'Gingerbread U':(4245.66,4500.00),
        'Homecoming':(266.31,150.00), 'K Playdate':(83.13,300.00),
        'K Orientation':(0.00,400.00), 'Milk & Cookies':(358.07,750.00),
        'Multicultural Night':(2022.51,2500.00), 'Outdoor Movie':(1884.20,2500.00),
        'Science Fair':(883.54,1500.00), 'Spring Fling':(0.00,2500.00),'Spring Dance K-5':(0.00,500.00),
        'Staff Appreciation':(3840.00,4000.00), 'Talent Show':(340.62,1000.00),
        'Talent Show DJ':(550.00,500.00), 'Volunteer Breakfast':(64.00,350.00),
        'Welcome Back Breakfast':(582.66,1000.00), 'WINGO':(0.00,500.00),
    },
    'Donations': {
        'BOE Gifts':(200.00,180.00), 'Cultural Arts':(14651.20,15000.00),
        'Folders':(580.00,600.00), 'Gardening':(0.00,500.00),
        'Hospitality':(124.70,750.00), '5th Staff T-Shirts':(191.78,125.00),
        'Recess Equipment':(1000.00,1000.00), 'School Spirit':(280.76,750.00),
        'Sling Bags':(2354.50,1500.00), 'Spelling Bee':(192.50,225.00),
        'Sunshine Fund':(210.00,500.00), 'Trick or Treat Street':(0.00,250.00),
        'WM Scholarships':(1000.00,1000.00),
    },
    'Grad Class Events': {
        'Monster Bash':(402.84,1300.00), 'Monster Bash DJ':(500.00,500.00),
        'Electric Parade':(0.00,900.00), 'Winter Social':(1167.86,1300.00),
        'Winter Social DJ':(500.00,500.00), 'Moving Up':(715.00,400.00),
        'Picnic':(2441.98,900.00),
    },
    'Grad Class Expenses': {
        'Graduating Class Gifts':(1034.41,500.00), 'Graduating Mural':(244.48,350.00),
        '5th Grade T-Shirts':(1494.25,800.00),
        'Trunk or Treat Fundraiser':(258.67,500.00),
        'The Night at Rinx':(710.00,650.00),
    },
}

def merge_actuals_into_budget(budget_dict, actuals_dict):
    # Apply QB name mapping first
    mapped_actuals = {}
    for qb_name, vals in actuals_dict.items():
        budget_name = QB_TO_BUDGET_MAP.get(qb_name, qb_name)
        if budget_name in mapped_actuals:
            # Sum if multiple QB items map to same budget line
            mapped_actuals[budget_name] = [
                mapped_actuals[budget_name][i] + vals[i] 
                for i in range(12)
            ]
        else:
            mapped_actuals[budget_name] = vals

    result = {}
    all_budget_items = {item for section in budget_dict.values() for item in section}
    for section, items in budget_dict.items():
        result[section] = {}
        for item, (last_yr, budget) in items.items():
            result[section][item] = (last_yr, budget, mapped_actuals.get(item, [0.0]*12))

    unmatched = [k for k in mapped_actuals if k not in all_budget_items]
    if unmatched:
        result['Other (from QuickBooks)'] = {}
        print(f'NOTE: {len(unmatched)} QB item(s) not matched to budget:')
        for item in unmatched:
            print(f'  - {item}')
            result['Other (from QuickBooks)'][item] = (0.0, 0.0, mapped_actuals[item])

    return result

INCOME_MERGED  = merge_actuals_into_budget(INCOME_BUDGET,  INCOME_ACTUALS_LIVE)
EXPENSE_MERGED = merge_actuals_into_budget(EXPENSE_BUDGET, EXPENSE_ACTUALS_LIVE)
print(f'Budget + actuals merged  |  Income sections: {len(INCOME_MERGED)}  |  Expense sections: {len(EXPENSE_MERGED)}')


Budget + actuals merged  |  Income sections: 5  |  Expense sections: 8


## Cell 7 — Excel Style Helpers

In [105]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

NAVY='1F3864'; TEAL='2E75B6'; LTBLUE='BDD7EE'; GOLD='FFD966'
WHITE='FFFFFF'; LGREY='F2F2F2'; GREEN='E2EFDA'; RED_BG='FCE4D6'

SUBHDR_FONT = Font(name='Arial',bold=True,color=WHITE,size=10)
BODY_FONT   = Font(name='Arial',size=10)
BOLD_FONT   = Font(name='Arial',bold=True,size=10)
TOTAL_FONT  = Font(name='Arial',bold=True,size=10,color=NAVY)

NAVY_FILL   = PatternFill('solid',fgColor=NAVY)
TEAL_FILL   = PatternFill('solid',fgColor=TEAL)
LTBLUE_FILL = PatternFill('solid',fgColor=LTBLUE)
GOLD_FILL   = PatternFill('solid',fgColor=GOLD)
LGREY_FILL  = PatternFill('solid',fgColor=LGREY)
GREEN_FILL  = PatternFill('solid',fgColor=GREEN)
RED_FILL    = PatternFill('solid',fgColor=RED_BG)

THIN        = Side(style='thin',   color='AAAAAA')
MED         = Side(style='medium', color=NAVY)
THIN_BORDER = Border(left=THIN,right=THIN,top=THIN,bottom=THIN)
MED_BORDER  = Border(left=MED, right=MED, top=MED, bottom=MED)
MONEY_FMT   = '$#,##0.00_);($#,##0.00)'

def sec_hdr(ws,label,row,n=4):
    ws.merge_cells(f'A{row}:{get_column_letter(n)}{row}')
    c=ws[f'A{row}']; c.value=label
    c.font=Font(name='Arial',bold=True,size=11,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='left',vertical='center',indent=1)
    ws.row_dimensions[row].height=20; return row+1

def col_hdrs(ws,row,labels):
    for i,lbl in enumerate(labels):
        c=ws.cell(row=row,column=i+1,value=lbl)
        c.font=SUBHDR_FONT; c.fill=TEAL_FILL; c.border=THIN_BORDER
        c.alignment=Alignment(horizontal='center' if i>0 else 'left',vertical='center',indent=1 if i==0 else 0)
    ws.row_dimensions[row].height=18; return row+1

def data_row(ws,row,label,amount,shade=False):
    fill=LGREY_FILL if shade else PatternFill()
    ws[f'A{row}'].value=label; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].fill=fill
    ws[f'A{row}'].border=THIN_BORDER; ws[f'A{row}'].alignment=Alignment(indent=2)
    ws[f'B{row}'].value=amount; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].fill=fill
    ws[f'B{row}'].border=THIN_BORDER; ws[f'B{row}'].number_format=MONEY_FMT
    ws[f'B{row}'].alignment=Alignment(horizontal='right')
    for col in ['C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
    ws.row_dimensions[row].height=16

def total_row(ws,row,label,val):
    for col in ['A','B','C','D']:
        ws[f'{col}{row}'].fill=LTBLUE_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value=label; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'B{row}'].value=val; ws[f'B{row}'].font=TOTAL_FONT
    ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right')
    ws.row_dimensions[row].height=18

print('Styles ready')


Styles ready


## Cell 8 — Sheet Builders

In [106]:
def build_treasurer(ws,qb,bank,month_label,org_name):
    ws.sheet_view.showGridLines=False
    for col,w in zip(['A','B','C','D'],[32,18,18,22]): ws.column_dimensions[col].width=w
    ws.merge_cells('A1:D1'); c=ws['A1']; c.value=org_name.upper()
    c.font=Font(name='Arial',bold=True,size=16,color=NAVY)
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=28
    ws.merge_cells('A2:D2'); c=ws['A2']
    c.value=f'Monthly Treasurer Report - {month_label}'
    c.font=Font(name='Arial',bold=True,size=12,color=TEAL); c.alignment=Alignment(horizontal='center')
    ws.merge_cells('A3:D3'); c=ws['A3']
    c.value=f'Generated: {datetime.today().strftime("%B %d, %Y")}'
    c.font=Font(name='Arial',italic=True,size=9,color='888888'); c.alignment=Alignment(horizontal='center')
    row=5; row=sec_hdr(ws,'INCOME',row); row=col_hdrs(ws,row,['Category','Amount','',''])
    inc_s=row
    for i,(k,v) in enumerate(qb['income'].items()): data_row(ws,row,k,v,shade=(i%2==1)); row+=1
    total_row(ws,row,'Total Income',f'=SUM(B{inc_s}:B{row-1})'); it=row; row+=2
    row=sec_hdr(ws,'EXPENSES',row); row=col_hdrs(ws,row,['Category','Amount','',''])
    exp_s=row
    for i,(k,v) in enumerate(qb['expenses'].items()): data_row(ws,row,k,v,shade=(i%2==1)); row+=1
    total_row(ws,row,'Total Expenses',f'=SUM(B{exp_s}:B{row-1})'); et=row; row+=2
    ws.merge_cells(f'A{row}:D{row}'); ws[f'A{row}'].value='NET INCOME / (LOSS)'
    ws[f'A{row}'].font=Font(name='Arial',bold=True,size=11,color=WHITE)
    ws[f'A{row}'].fill=NAVY_FILL; ws[f'A{row}'].alignment=Alignment(horizontal='left',indent=1); row+=1
    for col in ['A','B','C','D']: ws[f'{col}{row}'].fill=GOLD_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value='Net Income (Loss)'; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'B{row}'].value=f'=B{it}-B{et}'; ws[f'B{row}'].font=TOTAL_FONT
    ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right'); row+=2
    
    row=sec_hdr(ws,'BANK RECONCILIATION - Chase Business Checking',row)
    row=col_hdrs(ws,row,['Description','Amount','',''])
    for i,(lbl,amt) in enumerate([
        ('Beginning Balance',           bank['beginning_balance']),
        ('(+) Deposits & Additions',    bank['total_deposits']),
        ('(-) Checks Paid',            -bank['total_checks']),
        ('(-) Electronic Withdrawals', -bank.get('total_withdrawals', 0.0)),   # ← new
        ('(-) Fees',                   -bank['total_fees']),
    ]):
        data_row(ws,row,lbl,amt,shade=(i%2==1)); row+=1
    total_row(ws,row,'Ending Balance (per statement)',bank['ending_balance']); row+=1
    calc = (bank['beginning_balance']
            + bank['total_deposits']
            - bank['total_checks']
            - bank.get('total_withdrawals', 0.0)   # ← new
            - bank['total_fees'])
    diff = calc - bank['ending_balance']
    for lbl,val,fill in [
        ('Calculated Ending Balance', calc, PatternFill()),
        ('Difference (should be $0.00)', diff, GREEN_FILL if abs(diff)<0.01 else RED_FILL)
    ]:
        for col in ['A','B','C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=lbl; ws[f'A{row}'].font=BOLD_FONT; ws[f'A{row}'].alignment=Alignment(indent=2)
        ws[f'B{row}'].value=val; ws[f'B{row}'].font=BOLD_FONT
        ws[f'B{row}'].number_format=MONEY_FMT; ws[f'B{row}'].alignment=Alignment(horizontal='right')
        ws.row_dimensions[row].height=16; row+=1
    row+=1
    if bank['daily_balances']:
        row=sec_hdr(ws,'DAILY ENDING BALANCES',row)
        for col,hdr in zip(['A','B'],['Date','Balance']):
            ws[f'{col}{row}'].value=hdr; ws[f'{col}{row}'].font=SUBHDR_FONT
            ws[f'{col}{row}'].fill=TEAL_FILL; ws[f'{col}{row}'].border=THIN_BORDER
            ws[f'{col}{row}'].alignment=Alignment(horizontal='center')
        row+=1
        for i,(dt,bal) in enumerate(sorted(bank['daily_balances'].items())):
            fill=LGREY_FILL if i%2 else PatternFill()
            ws[f'A{row}'].value=dt; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].fill=fill
            ws[f'A{row}'].border=THIN_BORDER; ws[f'A{row}'].alignment=Alignment(horizontal='center')
            ws[f'B{row}'].value=bal; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].fill=fill
            ws[f'B{row}'].border=THIN_BORDER; ws[f'B{row}'].number_format=MONEY_FMT
            ws[f'B{row}'].alignment=Alignment(horizontal='right'); row+=1


def build_budget(ws,title,merged_data,org_name,fiscal_months,current_idx):
    ws.sheet_view.showGridLines=False
    ws.column_dimensions['A'].width=28; ws.column_dimensions['B'].width=13; ws.column_dimensions['C'].width=13
    for i in range(12): ws.column_dimensions[get_column_letter(4+i)].width=9
    ws.column_dimensions[get_column_letter(16)].width=11; ws.column_dimensions[get_column_letter(17)].width=11
    ws.merge_cells(f'A1:{get_column_letter(17)}1'); c=ws['A1']
    c.value=f'{org_name.upper()}  -  {title}'
    c.font=Font(name='Arial',bold=True,size=13,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=26
    ws.merge_cells(f'A2:{get_column_letter(17)}2'); c=ws['A2']
    c.value=(f'As of {datetime.today().strftime("%B %d, %Y")}  |  '
             f'Fiscal Year July 2025 - June 2026  |  '
             f'Active month: {fiscal_months[current_idx]}')
    c.font=Font(name='Arial',italic=True,size=9,color='666666'); c.alignment=Alignment(horizontal='center')
    headers=['Category','Last Year','Budget (Annual)']+fiscal_months+['Total','Profit/Loss']
    for ci,hdr in enumerate(headers,1):
        c=ws.cell(row=3,column=ci,value=hdr)
        is_active=(4<=ci<=15 and (ci-4)==current_idx)
        c.font=Font(name='Arial',bold=True,size=9,color=NAVY if is_active else WHITE)
        c.fill=GOLD_FILL if is_active else TEAL_FILL
        c.alignment=Alignment(horizontal='center' if ci>1 else 'left',vertical='center',wrap_text=True)
        c.border=THIN_BORDER
    ws.row_dimensions[3].height=30
    dr=4
    for section,items in merged_data.items():
        ws.merge_cells(f'A{dr}:{get_column_letter(17)}{dr}'); c=ws[f'A{dr}']; c.value=section
        c.font=Font(name='Arial',bold=True,size=10,color=WHITE); c.fill=NAVY_FILL
        c.alignment=Alignment(horizontal='left',vertical='center',indent=1)
        ws.row_dimensions[dr].height=18; ss=dr+1; dr+=1
        for ir,(item,(last_yr,budget,monthly)) in enumerate(items.items()):
            fill=LGREY_FILL if ir%2==1 else PatternFill()
            c=ws.cell(row=dr,column=1,value=item)
            c.font=BODY_FONT; c.fill=fill; c.alignment=Alignment(indent=2); c.border=THIN_BORDER
            for ci,v in [(2,last_yr),(3,budget)]:
                c=ws.cell(row=dr,column=ci,value=v if v else None)
                c.font=BODY_FONT; c.fill=fill; c.number_format=MONEY_FMT
                c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            for mi,v in enumerate(monthly):
                is_active=(mi==current_idx)
                c=ws.cell(row=dr,column=4+mi,value=v if v else None)
                c.font=Font(name='Arial',bold=is_active,size=10)
                c.fill=GOLD_FILL if (is_active and v) else (PatternFill('solid',fgColor='FFF9E6') if is_active else fill)
                c.number_format=MONEY_FMT; c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            c=ws.cell(row=dr,column=16,value=f'=SUM(D{dr}:O{dr})')
            c.font=BODY_FONT; c.fill=fill; c.number_format=MONEY_FMT
            c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            c=ws.cell(row=dr,column=17,value=f'=C{dr}-P{dr}' if budget else None)
            c.font=BODY_FONT; c.fill=fill; c.number_format=MONEY_FMT
            c.alignment=Alignment(horizontal='right'); c.border=THIN_BORDER
            ws.row_dimensions[dr].height=15; dr+=1
        se=dr-1; c=ws.cell(row=dr,column=1,value=f'Total {section}')
        c.font=TOTAL_FONT; c.fill=LTBLUE_FILL; c.alignment=Alignment(indent=1); c.border=MED_BORDER
        for ci in range(2,18):
            cl=get_column_letter(ci)
            c=ws.cell(row=dr,column=ci,value=f'=SUM({cl}{ss}:{cl}{se})')
            c.font=TOTAL_FONT; c.fill=LTBLUE_FILL; c.number_format=MONEY_FMT
            c.alignment=Alignment(horizontal='right'); c.border=MED_BORDER
        ws.row_dimensions[dr].height=18; dr+=2
    c=ws.cell(row=dr,column=1,value='GRAND TOTAL')
    c.font=Font(name='Arial',bold=True,size=11,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(indent=1); c.border=MED_BORDER
    for ci in range(2,18):
        cl=get_column_letter(ci)
        c=ws.cell(row=dr,column=ci,value=f'=SUMIF(A4:A{dr-1},"Total*",{cl}4:{cl}{dr-1})')
        c.font=Font(name='Arial',bold=True,size=10,color=WHITE); c.fill=NAVY_FILL
        c.number_format=MONEY_FMT; c.alignment=Alignment(horizontal='right'); c.border=MED_BORDER
    ws.row_dimensions[dr].height=20


def build_givebacks(ws,givebacks,bank,org_name):
    ws.sheet_view.showGridLines=False
    for col,w in zip(['A','B','C','D','E','F'],[30,18,12,16,11,25]): ws.column_dimensions[col].width=w
    ws.merge_cells('A1:F1'); c=ws['A1']
    c.value=f'{org_name.upper()}  -  Giveback Reconciliation'
    c.font=Font(name='Arial',bold=True,size=13,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=26
    ws.merge_cells('A2:F2'); c=ws['A2']
    c.value=f'Generated: {datetime.today().strftime("%B %d, %Y")}'
    c.font=Font(name='Arial',italic=True,size=9,color='666666'); c.alignment=Alignment(horizontal='center')
    row=4
    for col,hdr in zip(['A','B','C','D','E','F'],['Item','Category','Transactions','Amount','% of Total','Source File']):
        c=ws[f'{col}{row}']; c.value=hdr; c.font=SUBHDR_FONT; c.fill=TEAL_FILL
        c.alignment=Alignment(horizontal='center' if col!='A' else 'left'); c.border=THIN_BORDER
    ws.row_dimensions[row].height=18; row+=1
    ds=row; trn=ds+len(givebacks)
    for i,g in enumerate(givebacks):
        fill=LGREY_FILL if i%2==1 else PatternFill()
        for col in ['A','B','C','D','E','F']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=g['item']; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
        ws[f'B{row}'].value=g['category']; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].alignment=Alignment(horizontal='center')
        ws[f'C{row}'].value=g['count']; ws[f'C{row}'].font=BODY_FONT; ws[f'C{row}'].alignment=Alignment(horizontal='center')
        ws[f'D{row}'].value=g['total']; ws[f'D{row}'].font=BODY_FONT
        ws[f'D{row}'].number_format=MONEY_FMT; ws[f'D{row}'].alignment=Alignment(horizontal='right')
        ws[f'E{row}'].value=f'=D{row}/D{trn}'; ws[f'E{row}'].font=BODY_FONT
        ws[f'E{row}'].number_format='0.0%'; ws[f'E{row}'].alignment=Alignment(horizontal='center')
        ws[f'F{row}'].value=g.get('source_file',''); ws[f'F{row}'].font=BODY_FONT; ws[f'F{row}'].alignment=Alignment(indent=1)
        ws.row_dimensions[row].height=15; row+=1
    for col in ['A','B','C','D','E','F']: ws[f'{col}{row}'].fill=LTBLUE_FILL; ws[f'{col}{row}'].border=MED_BORDER
    ws[f'A{row}'].value='TOTAL'; ws[f'A{row}'].font=TOTAL_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)
    ws[f'C{row}'].value=f'=SUM(C{ds}:C{row-1})'; ws[f'C{row}'].font=TOTAL_FONT; ws[f'C{row}'].alignment=Alignment(horizontal='center')
    ws[f'D{row}'].value=f'=SUM(D{ds}:D{row-1})'; ws[f'D{row}'].font=TOTAL_FONT
    ws[f'D{row}'].number_format=MONEY_FMT; ws[f'D{row}'].alignment=Alignment(horizontal='right')
    ws[f'E{row}'].value='100.0%'; ws[f'E{row}'].font=TOTAL_FONT; ws[f'E{row}'].alignment=Alignment(horizontal='center')
    ws.row_dimensions[row].height=18; row+=2
    sec_hdr(ws,'GIVEBACK <-> BANK RECONCILIATION',row,n=6); row+=1
    gb_total=sum(g['total'] for g in givebacks)
    bank_gb=next((d['amount'] for d in bank['deposits']
                  if 'gb payout' in d.get('description','').lower()
                  or 'givebacks' in d.get('description','').lower()),0.0)
    for lbl,val in [('Givebacks Platform Total',gb_total),
                     ('Givebacks Deposit in Bank Statement',bank_gb),
                     ('Difference',gb_total-bank_gb)]:
        is_d=lbl=='Difference'
        fill=(GREEN_FILL if abs(gb_total-bank_gb)<0.01 else RED_FILL) if is_d else PatternFill()
        for col in ['A','B','C','D','E','F']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=lbl; ws[f'A{row}'].font=BOLD_FONT; ws[f'A{row}'].alignment=Alignment(indent=2)
        ws[f'D{row}'].value=val; ws[f'D{row}'].font=BOLD_FONT
        ws[f'D{row}'].number_format=MONEY_FMT; ws[f'D{row}'].alignment=Alignment(horizontal='right')
        ws.row_dimensions[row].height=16; row+=1


def build_manifest(ws,gb_folder,qb_folder,bank_folder,org_name,month_label,fiscal_idx,fiscal_months):
    ws.sheet_view.showGridLines=False
    for col,w in zip(['A','B','C','D'],[15,35,22,14]): ws.column_dimensions[col].width=w
    ws.merge_cells('A1:D1'); c=ws['A1']; c.value=f'{org_name.upper()}  -  File Manifest'
    c.font=Font(name='Arial',bold=True,size=13,color=WHITE); c.fill=NAVY_FILL
    c.alignment=Alignment(horizontal='center',vertical='center'); ws.row_dimensions[1].height=26
    ws.merge_cells('A2:D2'); c=ws['A2']
    c.value=(f'Report: {month_label}  |  Column: {fiscal_months[fiscal_idx]}  |  '
             f'Generated: {datetime.today().strftime("%B %d, %Y at %I:%M %p")}')
    c.font=Font(name='Arial',italic=True,size=9,color='666666'); c.alignment=Alignment(horizontal='center')
    row=4
    for col,hdr in zip(['A','B','C','D'],['Type','Filename','Last Modified','Size']):
        c=ws[f'{col}{row}']; c.value=hdr; c.font=SUBHDR_FONT; c.fill=TEAL_FILL
        c.alignment=Alignment(horizontal='left' if col=='B' else 'center'); c.border=THIN_BORDER
    ws.row_dimensions[row].height=18; row+=1
    all_files=([('Givebacks',f) for f in sorted(gb_folder.glob('*.csv'))]+
               [('QuickBooks',f) for f in sorted(qb_folder.glob('*.csv'))]+
               [('Bank Statement',f) for f in sorted(bank_folder.glob('*.pdf'))])
    for i,(ft,fp) in enumerate(all_files):
        fill=LGREY_FILL if i%2==1 else PatternFill()
        stat=fp.stat(); mod=datetime.fromtimestamp(stat.st_mtime).strftime('%Y-%m-%d %H:%M')
        size=f'{stat.st_size/1024:.1f} KB'
        for col in ['A','B','C','D']: ws[f'{col}{row}'].fill=fill; ws[f'{col}{row}'].border=THIN_BORDER
        ws[f'A{row}'].value=ft; ws[f'A{row}'].font=BODY_FONT; ws[f'A{row}'].alignment=Alignment(horizontal='center')
        ws[f'B{row}'].value=fp.name; ws[f'B{row}'].font=BODY_FONT; ws[f'B{row}'].alignment=Alignment(indent=1)
        ws[f'C{row}'].value=mod; ws[f'C{row}'].font=BODY_FONT; ws[f'C{row}'].alignment=Alignment(horizontal='center')
        ws[f'D{row}'].value=size; ws[f'D{row}'].font=BODY_FONT; ws[f'D{row}'].alignment=Alignment(horizontal='right',indent=1)
        ws.row_dimensions[row].height=16; row+=1
    ws.merge_cells(f'A{row}:D{row}')
    ws[f'A{row}'].value=f'Total files processed: {len(all_files)}'
    ws[f'A{row}'].font=BOLD_FONT; ws[f'A{row}'].alignment=Alignment(indent=1)

print('Sheet builders ready')


Sheet builders ready


## Cell 9 — Generate Excel Report

In [107]:
print(f'Building workbook  ->  {ORG_NAME}  |  {MONTH_LABEL}  |  Column: {FISCAL_MONTHS[FISCAL_IDX]}')
wb = openpyxl.Workbook()

ws1 = wb.active; ws1.title = 'Treasurer Report'
build_treasurer(ws1, qb, bank, MONTH_LABEL, ORG_NAME)
print('  Tab 1: Treasurer Report')

ws2 = wb.create_sheet('Income Budget vs Actuals')
build_budget(ws2, 'Budget vs Actuals - Income', INCOME_MERGED, ORG_NAME, FISCAL_MONTHS, FISCAL_IDX)
print(f'  Tab 2: Income Budget vs Actuals  (column {FISCAL_MONTHS[FISCAL_IDX]} updated from {QB_FILE.name})')

ws3 = wb.create_sheet('Expense Budget vs Actuals')
build_budget(ws3, 'Budget vs Actuals - Expenses', EXPENSE_MERGED, ORG_NAME, FISCAL_MONTHS, FISCAL_IDX)
print(f'  Tab 3: Expense Budget vs Actuals  (column {FISCAL_MONTHS[FISCAL_IDX]} updated from {QB_FILE.name})')

ws4 = wb.create_sheet('Giveback Reconciliation')
build_givebacks(ws4, givebacks, bank, ORG_NAME)
print('  Tab 4: Giveback Reconciliation')

ws5 = wb.create_sheet('File Manifest')
build_manifest(ws5, GB_FOLDER, QB_FOLDER, BANK_FOLDER, ORG_NAME, MONTH_LABEL, FISCAL_IDX, FISCAL_MONTHS)
print('  Tab 5: File Manifest')

wb.save(OUTPUT_FILE)
print(f'\nSaved -> {OUTPUT_FILE}')


Building workbook  ->  Setauket School PTA  |  April 2026  |  Column: APR
  Tab 1: Treasurer Report
  Tab 2: Income Budget vs Actuals  (column APR updated from quickbooks_april_2026.csv)
  Tab 3: Expense Budget vs Actuals  (column APR updated from quickbooks_april_2026.csv)
  Tab 4: Giveback Reconciliation
  Tab 5: File Manifest

Saved -> output/Treasurer_Report_April_2026.xlsx


## Cell 10 — Financial Summary

In [108]:
print('='*55)
print(f'  {ORG_NAME}  -  {MONTH_LABEL}')
print(f'  Budget column updated: {FISCAL_MONTHS[FISCAL_IDX]} (index {FISCAL_IDX})')
print('='*55)
print(f'  Income            : ${qb["income_total"]:>12,.2f}')
print(f'  Expenses          : ${qb["expense_total"]:>12,.2f}')
print(f'  Net Income (Loss) : ${qb["net_income"]:>12,.2f}')
print('-'*55)
print(f'  Bank Beginning    : ${bank["beginning_balance"]:>12,.2f}')
print(f'  Bank Ending       : ${bank["ending_balance"]:>12,.2f}')
calc = bank['beginning_balance']+bank['total_deposits']-bank['total_checks']-bank.get('total_withdrawals', 0.0)-bank['total_fees']
diff = calc - bank['ending_balance']
print(f'  Reconciliation    : {"Balanced" if abs(diff)<0.01 else f"Off by ${abs(diff):,.2f}"}')
print(f'  Givebacks Total   : ${sum(g["total"] for g in givebacks):>12,.2f}')
print('='*55)
hist_files = sorted(HISTORY_DIR.glob('*.json'))
print(f'\n  Months stored in history: {len(hist_files)}')
for hf in hist_files:
    e = json.loads(hf.read_text())
    fi = e.get("fiscal_index", "?")
    col = FISCAL_MONTHS[fi] if isinstance(fi, int) else "?"
    print(f'    [{col:<5}] {e["month_label"]:<20} income=${e["income_total"]:>10,.2f}  expenses=${e["expense_total"]:>10,.2f}')
print(f'\n  Output: {OUTPUT_FILE}')


  Setauket School PTA  -  April 2026
  Budget column updated: APR (index 9)
  Income            : $   31,732.00
  Expenses          : $   21,534.79
  Net Income (Loss) : $   10,197.21
-------------------------------------------------------
  Bank Beginning    : $   32,203.66
  Bank Ending       : $   38,056.59
  Reconciliation    : Balanced
  Givebacks Total   : $    7,761.00

  Months stored in history: 2
    [APR  ] April 2026           income=$ 31,732.00  expenses=$ 21,534.79
    [MAR  ] March 2026           income=$  8,405.21  expenses=$  2,552.28

  Output: output/Treasurer_Report_April_2026.xlsx


In [109]:
import os
hist_files = sorted(Path('data/history').glob('*.json'))
for hf in hist_files:
    print(f'{hf.name}  ({os.path.getsize(hf)} bytes)')
print('Expenses from QB detail:')
for k, v in qb['expenses'].items():
    print(f'  {k:<35} ${v:,.2f}')
print(f'\nTotal: ${qb["expense_total"]:,.2f}')    

April_2026.json  (935 bytes)
March_2026.json  (510 bytes)
Expenses from QB detail:
  Bank Charges & Fees                 $30.50
  Movie Night                         $668.94
  Scholastic Book Fairs               $5,841.48
  Spring Dance K-5                    $500.00
  Talent Show                         $273.92
  Arts In Education                   $4,045.00
  Basket dinner                       $5,905.54
  Basket Dinner - Venue               $3,214.25
  Milk & Cookies                      $362.19
  Spring Fling                        $335.00
  Staff Appreciation                  $134.24
  Welcome Back Staff                  $223.73

Total: $21,534.79


In [110]:
#old_hist = HISTORY_DIR / 'April_2026.json'
#f old_hist.exists():
#    old_hist.unlink()
#    print(f'Deleted: {old_hist}')
#else:
#    print('File not found')

In [95]:
#print('All transactions from QB detail:')
#for t in qb['transactions']:
#    print(f"  {t['date']}  {t['type']:<12}  {t['category']:<25}  "
#          f"{'INCOME' if t['is_income'] else 'EXPENSE'}  ${t['amount']:,.2f}  "
#          f"payee={t['payee']}")

## Cell 11 — Push to GitHub
Pushes code changes to your GitHub repository via SSH.

**What gets pushed:** notebook, README, .gitignore, requirements.txt — never input files, output Excel, or `.env`.

**One-time setup (run in terminal, not here):**
```bash
git init
git remote add origin git@github.com:yourname/pta-treasurer.git
# Add your SSH public key at: github.com → Settings → SSH and GPG keys
```


In [113]:
import subprocess

# Files/folders that should NEVER be pushed
SENSITIVE_PATTERNS = ['input/', 'output/', 'data/', '.env', '*.xlsx', '*.pdf', '*.csv','debug_']

def run_git(args, check=True):
    """Run a git command and return (returncode, stdout, stderr)."""
    result = subprocess.run(
        ['git'] + args,
        capture_output=True, text=True
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'git {" ".join(args)} failed:\n{result.stderr}')
    return result.returncode, result.stdout.strip(), result.stderr.strip()

def push_to_github(commit_message=None):
    print('GitHub Push')
    print('='*55)

    # 1. Check git is initialized
    code, _, _ = run_git(['rev-parse', '--git-dir'], check=False)
    if code != 0:
        print('ERROR: Not a git repository.')
        print('Run in terminal:')
        print('  git init')
        print(f'  git remote add {GITHUB_REMOTE} git@github.com:yourname/pta-treasurer.git')
        return

    # 2. Check remote exists
    _, remotes, _ = run_git(['remote'])
    if GITHUB_REMOTE not in remotes.split():
        print(f'ERROR: Remote "{GITHUB_REMOTE}" not found.')
        print(f'Run in terminal: git remote add {GITHUB_REMOTE} git@github.com:yourname/pta-treasurer.git')
        return

    # 3. Check .gitignore exists and protects sensitive files
    gitignore = Path('.gitignore')
    if not gitignore.exists():
        print('Creating .gitignore...')
        gitignore.write_text(
            '# Input files - never commit\n'
            'input/\n'
            'output/\n'
            'data/\n'
            '*.xlsx\n'
            '*.pdf\n'
            '*.csv\n'
            '.env\n'
            '__pycache__/\n'
            '*.pyc\n'
            '.DS_Store\n'
        )
        print('  .gitignore created')

    # 4. Show git status
    _, status, _ = run_git(['status', '--short'])
    if not status:
        print('Nothing to commit - working tree clean.')
        return

    print('\nFiles changed:')
    for line in status.split('\n'):
        if not line.strip(): continue
        parts = line.split(None, 1)
        if len(parts) < 2: continue
        flag  = parts[0].strip()
        fname = parts[1].strip()
        is_sensitive = any(
            fname.startswith(p.rstrip('*').rstrip('/')) or
            fname.endswith(p.lstrip('*'))
            for p in SENSITIVE_PATTERNS
        )
        icon = '  SKIP (sensitive)' if is_sensitive else '  will push'
        print(f'  {flag} {fname:<45} {icon}')

    # 5. Ask for commit message
    if commit_message is None:
        default_msg = f'Update report code - {MONTH_LABEL}'
        commit_message = input(f'\nCommit message [{default_msg}]: ').strip()
        if not commit_message:
            commit_message = default_msg

    # 6. Stage only safe files (not sensitive paths)
    print('\nStaging files...')
    safe_to_add = []
    for line in status.split('\n'):
        if not line.strip(): continue
        parts = line.split(None, 1)
        if len(parts) < 2: continue
        flag  = parts[0].strip()
        fname = parts[1].strip()
        if not fname: continue
        # Only stage modified and added files — NOT untracked (??)
        if flag not in ('M', 'A', 'AM', 'MM', 'R'): continue
        is_sensitive = any(
            fname.startswith(p.rstrip('*').rstrip('/')) or
            fname.endswith(p.lstrip('*'))
            for p in SENSITIVE_PATTERNS
        )
        if not is_sensitive:
            safe_to_add.append(fname)
        
    if not safe_to_add:
        print('No safe files to push (all changes are in sensitive paths).')
        return

    for f in safe_to_add:
        run_git(['add', f])
        print(f'  staged: {f}')

    # 7. Commit
    print(f'\nCommitting: "{commit_message}"')
    run_git(['commit', '-m', commit_message])

    # 8. Push
    print(f'Pushing to {GITHUB_REMOTE}/{GITHUB_BRANCH}...')
    run_git(['push', GITHUB_REMOTE, GITHUB_BRANCH])

    # 9. Show result
    _, log, _ = run_git(['log', '--oneline', '-1'])
    _, remote_url, _ = run_git(['remote', 'get-url', GITHUB_REMOTE])
    repo_url = remote_url.replace('git@github.com:', 'https://github.com/').replace('.git','')

    print('='*55)
    print(f'  Pushed successfully!')
    print(f'  Commit : {log}')
    print(f'  Repo   : {repo_url}')
    print('='*55)


# ── Run ───────────────────────────────────────────────────────────────────────
push_to_github()


GitHub Push

Files changed:
  M PTA_Treasurer_Report_v4.ipynb                   will push
  ?? data                                            SKIP (sensitive)
  ?? debug_after_login.png                           SKIP (sensitive)
  ?? debug_error.png                                 SKIP (sensitive)

Commit message [Update report code - April 2026]: Updates to Income and Expenses Categories

Staging files...
  staged: PTA_Treasurer_Report_v4.ipynb

Committing: "Updates to Income and Expenses Categories"
Pushing to origin/main...
  Pushed successfully!
  Commit : 4b923fa Updates to Income and Expenses Categories
  Repo   : https://github.com/deepssharma/pta_treasurer


In [24]:
import subprocess
result = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if line:
        print(repr(line))  # repr shows exact characters including spaces

' M PTA_Treasurer_Report_v4.ipynb'
'?? data'
'?? debug_after_login.png'
'?? debug_error.png'
'?? debug_login.png'
'?? debug_payouts.png'


In [25]:
_, status, _ = run_git(['status', '--short'])
for line in status.split('\n'):
    if not line.strip(): continue
    print(repr(line))
    print(f'  line[0]={repr(line[0])} line[1]={repr(line[1])} line[2]={repr(line[2])} line[3:]={repr(line[3:])}')

'M PTA_Treasurer_Report_v4.ipynb'
  line[0]='M' line[1]=' ' line[2]='P' line[3:]='TA_Treasurer_Report_v4.ipynb'
'?? data'
  line[0]='?' line[1]='?' line[2]=' ' line[3:]='data'
'?? debug_after_login.png'
  line[0]='?' line[1]='?' line[2]=' ' line[3:]='debug_after_login.png'
'?? debug_error.png'
  line[0]='?' line[1]='?' line[2]=' ' line[3:]='debug_error.png'
'?? debug_login.png'
  line[0]='?' line[1]='?' line[2]=' ' line[3:]='debug_login.png'
'?? debug_payouts.png'
  line[0]='?' line[1]='?' line[2]=' ' line[3:]='debug_payouts.png'
